# 레슨 06 — 브라우저 자동화 입문

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/06/%5B%ED%95%99%EC%83%9D%EC%9A%A9%5D%20%EB%A0%88%EC%8A%A8%2006%20%E2%80%94%20%EB%B8%8C%EB%9D%BC%EC%9A%B0%EC%A0%80%20%EC%9E%90%EB%8F%99%ED%99%94%20%EC%9E%85%EB%AC%B8.ipynb)

이 노트북은 읽기와 따라하기용 강의 노트북이다. 학생은 셀을 위에서 아래로 실행하며 웹 자동화에서 상태가 어떻게 유지되고 화면 조작이 어떤 순서로 기록되는지 확인한다. 브라우저 자동화 입문는 실제 사이트 대신 합성 fixture로 안전하게 연습한다.

## 학습 목표

1. 브라우저 자동화가 필요한 상황을 구분한다.
2. locator, fill, click, text_content 흐름을 이해한다.
3. 폼 입력과 버튼 클릭 결과를 검증한다.
4. 여러 CSV 케이스로 반복 QA를 수행한다.
5. 자동화 로그를 CSV로 저장한다.

---

## 1. 브라우저 자동화 흐름

브라우저 자동화는 page를 열고 locator로 요소를 찾고 fill/click으로 상태를 바꾼 뒤 결과를 확인하는 흐름이다. 실제 Playwright 문법과 같은 개념을 수업용 MiniPage로 재현한다.


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/06/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

class MiniLocator:
    def __init__(self, page, selector):
        self.page = page
        self.selector = selector
    def elements(self):
        return self.page.soup.select(self.selector)
    def count(self):
        return len(self.elements())
    def first(self):
        items = self.elements()
        if not items:
            raise ValueError(f'no element for {self.selector}')
        return items[0]
    def text_content(self):
        return self.first().get_text(' ', strip=True)
    def all_text_contents(self):
        return [el.get_text(' ', strip=True) for el in self.elements()]
    def get_attribute(self, name):
        return self.first().get(name)
    def fill(self, value):
        self.first()['value'] = str(value)
    def click(self):
        return self.page._click(self.first())

class MiniPage:
    def __init__(self, html):
        self.soup = BeautifulSoup(html, 'html.parser')
        self.step = 0
        self.log = []
    def locator(self, selector):
        return MiniLocator(self, selector)
    def text_content(self, selector):
        return self.locator(selector).text_content()
    def fill(self, selector, value):
        self.locator(selector).fill(value)
        self.log.append({'action': 'fill', 'selector': selector, 'value': str(value)})
    def click(self, selector):
        result = self.locator(selector).click()
        self.log.append({'action': 'click', 'selector': selector, 'result': result})
        return result
    def _value(self, selector):
        el = self.soup.select_one(selector)
        return '' if el is None else el.get('value', '')
    def _click(self, el):
        action = el.get('data-action', '')
        if action == 'submit-profile':
            name = self._value('#student-name')
            course = self._value('#course-name')
            memo = self._value('#memo')
            out = self.soup.select_one('#result')
            out.string = f'{name} / {course} / {memo}'
            out['data-state'] = 'submitted'
            return 'submitted'
        if action == 'toggle-complete':
            target = self.soup.select_one(el.get('data-target', ''))
            if target:
                target['data-status'] = 'done' if target.get('data-status') != 'done' else 'pending'
                return target['data-status']
        if action == 'open-tab':
            target_id = el.get('data-target')
            for panel in self.soup.select('[role="tabpanel"]'):
                panel['hidden'] = 'true'
            target = self.soup.select_one(f'#{target_id}')
            if target and target.has_attr('hidden'):
                del target['hidden']
            return target_id
        return action or 'clicked'
    def visible_elements(self, selector):
        items = []
        for el in self.soup.select(selector):
            delay = int(el.get('data-delay-step', '0'))
            hidden = el.has_attr('hidden') or el.get('aria-hidden') == 'true'
            if delay <= self.step and not hidden:
                items.append(el)
        return items
    def tick(self):
        self.step += 1
        return self.step
    def wait_for_selector(self, selector, timeout_steps=5):
        for _ in range(timeout_steps + 1):
            items = self.visible_elements(selector)
            if items:
                return items[0]
            self.tick()
        raise TimeoutError(f'timeout waiting for {selector}')

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

page = MiniPage(load_text('form_page.html'))
print(page.text_content('h1'))


---

## 2. 폼 입력과 클릭

입력은 value 상태를 바꾸고 클릭은 output 상태를 바꾼다.


In [ ]:
page.fill('#student-name', '김도윤')
page.fill('#course-name', 'Python')
page.fill('#memo', '첫 자동화')
page.click('#submit-profile')
print(page.text_content('#result'))


---

## 3. 반복 QA

CSV 케이스를 이용하면 같은 폼을 여러 입력으로 반복 검증할 수 있다.


In [ ]:
cases = list(csv.DictReader(load_text('form_cases.csv').splitlines()))
outputs = []
for row in cases:
    p = MiniPage(load_text('form_page.html'))
    p.fill('#student-name', row['name'])
    p.fill('#course-name', row['course'])
    p.fill('#memo', row['memo'])
    p.click('#submit-profile')
    outputs.append(p.text_content('#result'))
print(outputs[:2])


---

## 데이터 출처와 안전 규칙

이 레슨의 파일은 모두 수업용 합성 데이터다. 실제 사이트의 개인정보, 로그인 정보, 유료 콘텐츠를 포함하지 않는다. 실제 사이트로 확장할 때는 robots.txt, 이용 약관, 요청 간격, 개인정보 여부를 먼저 확인한다. 수업 중에는 fixture를 반복 실행하며 구조를 익히고, 외부 사이트를 빠르게 반복 요청하지 않는다.

---

## 수업 운영 메모

이 절은 수업 중 교사가 질문으로 풀어낼 수 있는 운영형 설명이다. 학생이 셀을 실행한 뒤 결과만 맞히지 않고 자동화 절차를 말로 설명하도록 돕는다.

### 1. 브라우저 자동화가 필요한 순간

정적 HTML 파싱으로 해결되는 문제에 브라우저 자동화를 쓰면 느리고 불안정해진다. 반대로 클릭, 입력, 탭 전환, 동적 결과 확인이 필요하면 브라우저 모델이 더 자연스럽다.

### 2. MiniPage의 목적

이 레슨의 MiniPage는 실제 Playwright를 그대로 흉내 내는 도구가 아니라 클릭과 입력의 순서를 안전하게 연습하는 모델이다. 외부 브라우저 설치 실패 없이 수업 흐름을 유지하기 위한 장치다.

### 3. 입력값과 결과 영역

폼 자동화에서 학생이 자주 놓치는 부분은 입력 후 결과 영역이 정말 바뀌었는지 확인하지 않는 것이다. fill, click, result 확인을 한 묶음으로 작성하게 한다.

### 4. 탭과 상태

탭 전환은 보이는 화면만 바꾸는 것이 아니라 hidden 속성이나 aria 상태를 조정한다. 학생에게 현재 보이는 panel이 무엇인지 selector로 확인하게 하면 UI 자동화의 구조를 이해할 수 있다.

### 5. 로그의 역할

브라우저 자동화는 화면이 바뀌기 때문에 나중에 어떤 버튼을 눌렀는지 기억하기 어렵다. action, selector, value, result를 로그로 남기면 실패 재현이 쉬워진다.

### 6. 실제 도구로 확장

Playwright나 Selenium으로 넘어가도 개념은 같다. locator를 잡고, 값을 채우고, 클릭하고, 결과를 기다리고, 실패를 기록하는 순서만 유지하면 도구 문법은 바뀌어도 흐름은 유지된다.

### 7. 채점 관점

학생 답안은 버튼을 누른 결과만 보지 말고 클릭 전후 상태와 로그 길이를 함께 본다. 자동화는 보이지 않는 절차가 중요하므로 결과 문자열 하나만 맞아도 절차가 비어 있으면 보완한다.

## 실무 전환 기준

MiniPage는 실제 브라우저를 실행하지 않는 수업용 모델이다. 그래도 실제 Playwright나 Selenium으로 넘어갈 때 필요한 사고방식은 그대로 담고 있다. 먼저 대상 페이지를 열고, 안정적인 locator를 고르고, 값을 입력하고, 클릭 후 결과 상태를 확인하고, 실패했을 때 재현할 수 있는 로그를 남긴다. 이 순서가 흔들리면 도구를 바꿔도 자동화가 안정되지 않는다.

정적 HTML 파싱으로 충분한 작업인지, 실제 브라우저가 필요한 작업인지도 구분해야 한다. 단순 표 추출은 BeautifulSoup이 빠르고 단순하다. 반면 로그인 후 화면, 버튼 클릭 결과, 탭 전환, 지연 로딩, 모달 확인은 브라우저 자동화가 필요하다. 학생이 어떤 도구를 써야 할지 스스로 판단하게 하려면 “화면 상태를 바꿔야 하는가”라는 질문을 먼저 던지게 한다.

## 디버깅 루틴

브라우저 자동화가 실패하면 무작정 코드를 길게 고치지 않는다. 첫째, selector가 몇 개의 요소를 잡는지 count로 확인한다. 둘째, fill 이후 value가 들어갔는지 확인한다. 셋째, click 이후 result나 data-state가 바뀌었는지 확인한다. 넷째, wait가 필요한 화면인지 판단한다. 마지막으로 log를 CSV로 저장해 어떤 action이 실행되었는지 본다.

수업에서는 실패 상황도 일부러 보여주는 편이 좋다. 존재하지 않는 selector를 넣으면 MiniLocator가 ValueError를 발생시킨다. 이 오류는 학생에게 selector 검증의 필요성을 보여준다. 실제 Playwright에서는 timeout이나 strict mode 오류로 나타날 수 있으므로, 오류 메시지를 두려워하기보다 실패 위치를 좁히는 자료로 보게 한다.

## 안전 기준

이번 레슨의 fixture는 실제 학생 정보가 들어 있지 않은 합성 데이터다. 실제 사이트로 확장할 때는 테스트 계정을 사용하고, 비밀번호나 개인정보를 코드와 로그에 남기지 않는다. 반복 자동화는 요청 간격과 최대 재시도 횟수를 제한한다. 특히 수업 중 여러 학생이 같은 외부 사이트를 동시에 요청하면 서비스에 부담을 줄 수 있으므로, 합성 fixture에서 흐름을 먼저 충분히 연습한 뒤 제한적으로 확장한다.

## 수업 중 확인 질문

- 이 문제는 정적 파싱으로 충분한가, 클릭이 필요한가?
- selector가 화면 문구에 의존하고 있지는 않은가?
- 입력 후 value를 확인했는가?
- 클릭 후 상태가 실제로 바뀌었는가?
- 반복 실행 결과를 어떤 파일로 남겼는가?
- 실제 사이트로 옮길 때 개인정보와 요청 간격을 어떻게 보호할 것인가?

## selector 안티패턴과 대체 전략

브라우저 자동화에서 가장 빨리 깨지는 코드는 “지금 눈에 보이는 모양”에 지나치게 의존하는 코드다. 예를 들어 버튼의 색상 class, 화면에 표시된 전체 문장, 리스트의 현재 순번만 보고 selector를 잡으면 디자인 수정이나 번역 변경에 약하다. 반대로 id, data-testid, role, name, data-target처럼 기능적 의미가 있는 속성은 상대적으로 오래 유지된다. 수업에서는 학생에게 같은 요소를 class, text, id, data-testid로 각각 잡아 보게 한 뒤 어떤 선택이 유지보수에 유리한지 비교하게 한다.

폼 입력에서는 label 텍스트가 바뀌어도 id가 유지되는 경우가 많다. Todo 항목에서는 버튼 문구가 모두 “완료”로 같기 때문에 text 기반 selector는 어떤 항목을 누를지 모호하다. 이때 `button[data-target="#task-1"]`처럼 target을 명확히 지정하면 클릭 대상과 변경 대상의 관계가 코드에 남는다. 탭 UI에서도 버튼의 화면 문구보다 `data-target`과 panel id를 맞추는 편이 더 설명 가능하다.

## 자동화 실패를 읽는 방법

실패는 대부분 네 가지 중 하나다. 첫째, 파일을 못 읽었다. 이 경우 DATA_BASE와 파일 이름을 먼저 확인한다. 둘째, selector가 요소를 못 찾았다. 이 경우 count를 출력하고 HTML에서 속성 이름을 다시 본다. 셋째, 입력은 되었지만 클릭 결과가 바뀌지 않았다. 이 경우 버튼의 data-action과 target을 확인한다. 넷째, 결과가 늦게 나타났다. 이 경우 wait 조건을 잡아야 한다.

실제 브라우저 도구에서는 실패 메시지가 더 길고 복잡하다. TimeoutError, strict mode violation, element is not visible, detached from DOM 같은 메시지가 나올 수 있다. 이 레슨에서는 ValueError와 TimeoutError 정도만 다루지만, 학생에게 오류 메시지를 “망했다”가 아니라 “어느 단계가 깨졌는지 알려주는 단서”로 읽게 해야 한다.

## Playwright 문법으로 옮겨 보기

MiniPage에서 익힌 흐름은 Playwright로 거의 그대로 옮길 수 있다. `MiniPage(load_text(...))`는 실제 코드에서 `page.goto(url)`에 해당한다. `page.locator('#student-name').count()`는 Playwright에서도 같은 사고방식으로 사용한다. `page.fill('#student-name', '김도윤')`은 `await page.fill('#student-name', '김도윤')` 또는 `await page.locator('#student-name').fill('김도윤')`로 바뀐다. `page.click('#submit-profile')`도 실제로는 비동기 await가 붙을 뿐 흐름은 같다.

다만 실제 도구에서는 화면 렌더링과 네트워크가 개입한다. 그래서 클릭 직후 바로 결과를 읽지 않고 `await expect(page.locator('#result')).toContainText(...)`처럼 기대 상태를 기다린다. 이번 수업의 `wait_for_selector`는 이 사고방식을 간단히 흉내 낸 것이다. 학생이 이 연결을 이해하면 다음 단계에서 Playwright 문법을 배울 때 부담이 줄어든다.

## 로그 설계 예시

자동화 로그에는 너무 많은 정보를 넣지 않는다. 기본은 action, selector, value, result 네 가지면 충분하다. value에는 개인정보나 비밀번호를 넣지 않도록 주의한다. 폼 입력 수업에서는 합성 이름과 과정만 저장하지만, 실제 서비스에서는 사용자를 식별할 수 있는 값이 들어갈 수 있으므로 마스킹하거나 제외해야 한다. 실패 로그에는 row 번호, selector, error message 정도를 남기면 재현에 도움이 된다.

CSV 로그는 사람이 열어 보기 쉽고, JSON 요약은 프로그램이 읽기 좋다. 수업에서는 두 산출물의 역할을 구분하게 한다. CSV에는 케이스별 결과를 넣고, JSON에는 전체 케이스 수, 성공 수, 실패 수, 저장 경로 같은 요약값을 넣는다. 이 구분이 있어야 자동화 결과를 선생님이나 운영자가 빠르게 판단할 수 있다.

## 운영 자동화로 볼 수 없는 코드

한 번 실행해서 화면에 원하는 글자가 나왔다고 모두 자동화가 아니다. 입력 데이터가 코드에만 박혀 있고, 실패했을 때 어떤 selector에서 막혔는지 알 수 없고, 저장 산출물이 없으면 운영 자동화로 보기 어렵다. 또한 실제 사이트에 요청 간격 없이 반복 접근하거나, 학생 개인정보를 로그에 남기는 코드도 수업 목표와 맞지 않는다.

좋은 답안은 짧아도 절차가 분명하다. 파일을 읽고, 대상 요소를 찾고, 입력하고, 클릭하고, 결과를 검증하고, 로그를 남긴다. 이 여섯 단계가 보이면 코드 스타일이 조금 달라도 통과로 볼 수 있다. 반대로 출력 문자열 하나만 맞추고 중간 상태가 모두 빠져 있으면 보완하도록 안내한다.

## 제출물 리뷰 기준

학생 제출물을 볼 때는 정답 문자열보다 자동화 절차를 먼저 본다. `form_cases.csv`의 10개 케이스를 모두 처리했는지, 각 케이스마다 새 page를 만들었는지, result와 data-state를 함께 저장했는지 확인한다. Todo와 탭은 클릭 결과가 문자열로 출력되어도 실제 target 상태나 panel 텍스트를 다시 읽어야 한다. 이 과정을 빠뜨린 답안은 우연히 맞은 출력일 수 있다.

또한 산출물 파일을 열었을 때 사람이 이해할 수 있어야 한다. CSV header가 없거나 ok 값만 있고 어떤 입력 케이스인지 모르면 운영 로그로 쓰기 어렵다. JSON 요약에는 전체 건수, 성공 수, 실패 수, 주요 상태 전환 결과가 들어가야 한다. 이런 기준을 초반부터 잡아두면 이후 실제 브라우저 자동화 프로젝트에서 코드가 길어져도 검수 방식이 흔들리지 않는다.


---

# 레슨 06 — 실습 문제


브라우저 자동화 입문 레슨의 학생용 문제 노트북이다. 강의 노트북을 먼저 실행한 뒤 빈칸을 직접 채운다.

## 통과 기준

- 총 15문제 중 12문제 이상 정상 출력이면 통과.
- 문제 1~5는 구조 확인, 6~10은 상태 변화와 반복 처리, 11~15는 로그와 저장이다.
- 정답값은 적지 않는다. 출력 형태와 fixture 구조를 보고 직접 판단한다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/06/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

class MiniLocator:
    def __init__(self, page, selector):
        self.page = page
        self.selector = selector
    def elements(self):
        return self.page.soup.select(self.selector)
    def count(self):
        return len(self.elements())
    def first(self):
        items = self.elements()
        if not items:
            raise ValueError(f'no element for {self.selector}')
        return items[0]
    def text_content(self):
        return self.first().get_text(' ', strip=True)
    def all_text_contents(self):
        return [el.get_text(' ', strip=True) for el in self.elements()]
    def get_attribute(self, name):
        return self.first().get(name)
    def fill(self, value):
        self.first()['value'] = str(value)
    def click(self):
        return self.page._click(self.first())

class MiniPage:
    def __init__(self, html):
        self.soup = BeautifulSoup(html, 'html.parser')
        self.step = 0
        self.log = []
    def locator(self, selector):
        return MiniLocator(self, selector)
    def text_content(self, selector):
        return self.locator(selector).text_content()
    def fill(self, selector, value):
        self.locator(selector).fill(value)
        self.log.append({'action': 'fill', 'selector': selector, 'value': str(value)})
    def click(self, selector):
        result = self.locator(selector).click()
        self.log.append({'action': 'click', 'selector': selector, 'result': result})
        return result
    def _value(self, selector):
        el = self.soup.select_one(selector)
        return '' if el is None else el.get('value', '')
    def _click(self, el):
        action = el.get('data-action', '')
        if action == 'submit-profile':
            name = self._value('#student-name')
            course = self._value('#course-name')
            memo = self._value('#memo')
            out = self.soup.select_one('#result')
            out.string = f'{name} / {course} / {memo}'
            out['data-state'] = 'submitted'
            return 'submitted'
        if action == 'toggle-complete':
            target = self.soup.select_one(el.get('data-target', ''))
            if target:
                target['data-status'] = 'done' if target.get('data-status') != 'done' else 'pending'
                return target['data-status']
        if action == 'open-tab':
            target_id = el.get('data-target')
            for panel in self.soup.select('[role="tabpanel"]'):
                panel['hidden'] = 'true'
            target = self.soup.select_one(f'#{target_id}')
            if target and target.has_attr('hidden'):
                del target['hidden']
            return target_id
        return action or 'clicked'
    def visible_elements(self, selector):
        items = []
        for el in self.soup.select(selector):
            delay = int(el.get('data-delay-step', '0'))
            hidden = el.has_attr('hidden') or el.get('aria-hidden') == 'true'
            if delay <= self.step and not hidden:
                items.append(el)
        return items
    def tick(self):
        self.step += 1
        return self.step
    def wait_for_selector(self, selector, timeout_steps=5):
        for _ in range(timeout_steps + 1):
            items = self.visible_elements(selector)
            if items:
                return items[0]
            self.tick()
        raise TimeoutError(f'timeout waiting for {selector}')

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 — 폼 페이지 제목 읽기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
page = MiniPage(load_text('____'))
print(page.text_content('____'))


---

## 문제 2 — input 개수 세기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
print(page.locator('____').count())


---

## 문제 3 — 이름 입력하기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
page.fill('____', '김도윤')
print(page.locator('____').get_attribute('____'))


---

## 문제 4 — 과정과 메모 입력하기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
page.fill('____', 'Python')
page.fill('____', '첫 자동화')
print(page.locator('#course-name').get_attribute('value'))
print(page.locator('#memo').get_attribute('value'))


---

## 문제 5 — 저장 버튼 클릭하기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
result = page.click('____')
print(result)
print(page.text_content('____'))


---

## 문제 6 — data-testid selector 사용하기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
page2 = MiniPage(load_text('form_page.html'))
page2.fill('[data-testid="____"]', '이서연')
print(page2.locator('[data-testid="student-name"]').get_attribute('____'))


---

## 문제 7 — CSV 케이스 읽기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
cases = list(csv.DictReader(load_text('____').splitlines()))
print(len(cases))
print(cases[0]['____'])


---

## 문제 8 — 여러 케이스 폼 자동화

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
outputs = []
for row in cases:
    p = MiniPage(load_text('form_page.html'))
    p.fill('#student-name', row['____'])
    p.fill('#course-name', row['____'])
    p.fill('#memo', row['____'])
    p.click('#submit-profile')
    outputs.append(p.text_content('____'))
print(outputs[:2])


---

## 문제 9 — Todo 앱 상태 세기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
todo = MiniPage(load_text('____'))
items = todo.locator('li').elements()
counts = {}
for item in items:
    status = item['____']
    counts[status] = counts.get(status, 0) + 1
print(counts)


---

## 문제 10 — Todo 완료 버튼 클릭

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
print(todo.locator('____').get_attribute('data-status'))
todo.click('____')
print(todo.locator('____').get_attribute('data-status'))


---

## 문제 11 — 탭 전환하기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
tabs = MiniPage(load_text('____'))
print(tabs.text_content('#panel-python'))
tabs.click('button[data-target="____"]')
print(tabs.text_content('#panel-web'))


---

## 문제 12 — 행동 로그 확인

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
log_page = MiniPage(load_text('form_page.html'))
log_page.fill('#student-name', '로그학생')
log_page.click('#submit-profile')
print(log_page.____)


---

## 문제 13 — 실패 selector 처리

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
try:
    page.locator('____').text_content()
except Exception as error:
    print(type(error).__name__)


---

## 문제 14 — QA 결과 리스트 만들기

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
qa_rows = []
for text in outputs:
    qa_rows.append({'result': text, 'ok': ' / ' in text})
print(qa_rows[0])
print(sum(row['____'] for row in qa_rows))


---

## 문제 15 — QA 로그 CSV 저장

지시된 값을 코드로 추출하거나 상태를 변경한다.

**기대 결과 형태**: 요구한 값이 한 줄 또는 리스트/딕셔너리 형태로 출력된다.

**빈칸 힌트**: HTML 구조, CSV 헤더, 이전 예제를 확인한 뒤 `____` 부분을 직접 채운다.


In [ ]:
with open('lesson06_qa_log.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['result', 'ok'])
    writer.____()
    writer.____(qa_rows)
print('saved:', len(qa_rows))


---

## 제출 전 점검

문제를 다 푼 뒤에는 출력만 보지 말고 실행 순서를 다시 확인한다. 폼 자동화는 파일 읽기, selector 확인, 입력, 클릭, 결과 검증 순서로 진행되어야 한다. Todo와 탭 문제는 클릭 전후 같은 대상을 비교했는지 확인한다. 로그와 CSV 저장 문제는 파일 경로가 출력되고 실제 파일이 생성되는지까지 확인한다.

| 항목 | 확인 질문 | 기준 |
|---|---|---|
| selector | id나 data-testid를 사용했는가 | 화면 문구보다 구조 속성 우선 |
| 입력 | fill 이후 value를 확인했는가 | 입력값이 같은 selector에 남아 있음 |
| 클릭 | 클릭 후 result나 data-state를 읽었는가 | 버튼 동작이 상태 변화로 확인됨 |
| 반복 | CSV 모든 행을 처리했는가 | outputs 길이와 cases 길이 일치 |
| 상태 | Todo 클릭 전후 상태가 달라졌는가 | 같은 task id로 비교 |
| 탭 | target panel의 텍스트를 읽었는가 | Web 또는 Report panel 확인 |
| wait | 조건 selector를 기다렸는가 | 무작정 sleep을 쓰지 않음 |
| 저장 | CSV header와 row가 있는가 | 파일 패널에서 확인 가능 |

실제 브라우저 자동화로 확장할 때는 바로 외부 사이트를 반복 요청하지 않는다. 먼저 fixture에서 흐름을 검증하고, 테스트 계정과 요청 간격, 개인정보 저장 여부를 확인한다. 수업 제출물에는 실제 비밀번호나 개인 정보가 들어가지 않아야 한다.

---

# 레슨 06 — 최종 미션


폼 입력, Todo 클릭, 탭 전환을 자동화하고 QA 로그를 만든다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/06/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

class MiniLocator:
    def __init__(self, page, selector):
        self.page = page
        self.selector = selector
    def elements(self):
        return self.page.soup.select(self.selector)
    def count(self):
        return len(self.elements())
    def first(self):
        items = self.elements()
        if not items:
            raise ValueError(f'no element for {self.selector}')
        return items[0]
    def text_content(self):
        return self.first().get_text(' ', strip=True)
    def all_text_contents(self):
        return [el.get_text(' ', strip=True) for el in self.elements()]
    def get_attribute(self, name):
        return self.first().get(name)
    def fill(self, value):
        self.first()['value'] = str(value)
    def click(self):
        return self.page._click(self.first())

class MiniPage:
    def __init__(self, html):
        self.soup = BeautifulSoup(html, 'html.parser')
        self.step = 0
        self.log = []
    def locator(self, selector):
        return MiniLocator(self, selector)
    def text_content(self, selector):
        return self.locator(selector).text_content()
    def fill(self, selector, value):
        self.locator(selector).fill(value)
        self.log.append({'action': 'fill', 'selector': selector, 'value': str(value)})
    def click(self, selector):
        result = self.locator(selector).click()
        self.log.append({'action': 'click', 'selector': selector, 'result': result})
        return result
    def _value(self, selector):
        el = self.soup.select_one(selector)
        return '' if el is None else el.get('value', '')
    def _click(self, el):
        action = el.get('data-action', '')
        if action == 'submit-profile':
            name = self._value('#student-name')
            course = self._value('#course-name')
            memo = self._value('#memo')
            out = self.soup.select_one('#result')
            out.string = f'{name} / {course} / {memo}'
            out['data-state'] = 'submitted'
            return 'submitted'
        if action == 'toggle-complete':
            target = self.soup.select_one(el.get('data-target', ''))
            if target:
                target['data-status'] = 'done' if target.get('data-status') != 'done' else 'pending'
                return target['data-status']
        if action == 'open-tab':
            target_id = el.get('data-target')
            for panel in self.soup.select('[role="tabpanel"]'):
                panel['hidden'] = 'true'
            target = self.soup.select_one(f'#{target_id}')
            if target and target.has_attr('hidden'):
                del target['hidden']
            return target_id
        return action or 'clicked'
    def visible_elements(self, selector):
        items = []
        for el in self.soup.select(selector):
            delay = int(el.get('data-delay-step', '0'))
            hidden = el.has_attr('hidden') or el.get('aria-hidden') == 'true'
            if delay <= self.step and not hidden:
                items.append(el)
        return items
    def tick(self):
        self.step += 1
        return self.step
    def wait_for_selector(self, selector, timeout_steps=5):
        for _ in range(timeout_steps + 1):
            items = self.visible_elements(selector)
            if items:
                return items[0]
            self.tick()
        raise TimeoutError(f'timeout waiting for {selector}')

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


## 제출 산출물

- 실행 가능한 노트북
- 결과 CSV 또는 정리 파일
- 자동화 결과 요약 3문장
- 안전 규칙 점검 메모 2개

## 스타터 코드


In [ ]:
# CSV 케이스를 반복 실행해 qa_rows를 만들고 저장한다
qa_rows = []
# TODO


## 자동화 결과 요약

- 자동화 대상:
- 핵심 결과:
- 다음 실행 때 조심할 점:
